# Outcome overview

The outcome of the seizure-type cohort: every patient with the outcome and pre-op seizure types recorded. This is the cohort the improvement analyses start from (`1_improvement`, and `2_disappearance/disappearance_and_improvement.ipynb`).

Every table and figure is shown for both outcomes, Engel then ILAE (see `cc_stats/outcomes.py`): improved = Engel I–III, or ILAE 1–4 (at least a 50% reduction in seizure days). Both are assessed at the post-op neurology note at least 6 months after surgery.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cc_stats.outcomes import OUTCOMES, heading
from cc_stats.paths import ANALYSIS_CSV, DATASET
from cc_stats.plots import INK, PAIR_A, PAIR_B, save_figure

analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
has_cols = [c for c in analysis.columns if c.startswith('has_')]
print(f'dataset {DATASET}')

# One cohort per outcome: that outcome and pre-op seizure types recorded
cohorts = {outcome: analysis.dropna(subset=[O['improved']] + has_cols) for outcome, O in OUTCOMES.items()}
for outcome, cohort in cohorts.items():
    print(f"{OUTCOMES[outcome]['name']}: {len(cohort)} patients with the outcome and pre-op seizure types recorded")

## Patient flow

From everyone in the study to the patients each analysis uses, for each outcome: the outcome recorded, then pre-op seizure types and LGS (the improvement analyses), then pre- and post-op seizure types and LGS (the disappearance and change analyses).

In [ ]:
post_cols = [c for c in analysis.columns if c.startswith('had_post_')]


def flow_steps(outcome_col):
    """Patients at each step of the cohort, for one outcome."""
    outcome = analysis[outcome_col].notna()
    return [int(outcome.sum()),
            int((outcome & analysis[has_cols + ['lgs']].notna().all(axis=1)).sum()),
            int((outcome & analysis[has_cols + post_cols + ['lgs']].notna().all(axis=1)).sum())]


STEPS = ['with an {} outcome', 'with pre-op seizure types and LGS', 'with pre- and post-op seizure types and LGS']

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 1)
ax.set_ylim(0.02, 1)
ax.axis('off')
box = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=PAIR_A, linewidth=1.8)
arrow = dict(arrowstyle='-|>', color=INK, lw=1.2)
ax.text(0.5, 0.93, f'{len(analysis)} patients in the study', ha='center', va='center', fontsize=12, bbox=box)
for x, name, col in ((0.25, 'Engel', 'engel_class'), (0.75, 'ILAE', 'ilae')):
    top = (0.5, 0.87)
    for step, (n, label) in enumerate(zip(flow_steps(col), STEPS)):
        y = 0.70 - step * 0.25
        ax.annotate('', xy=(x, y + 0.05), xytext=top, arrowprops=arrow)
        ax.text(x, y, f'{n} {label.format(name)}', ha='center', va='center', fontsize=10.5, bbox=box)
        top = (x, y - 0.05)
ax.set_title('Patient flow', loc='left', fontsize=13)
save_figure('0_cohort', 'patient_flow')
plt.show()

## Improved vs not improved

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    outcome_counts = cohorts[outcome][O['improved']].map({1: 'improved', 0: 'not improved'}).value_counts()
    display(outcome_counts)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.pie(outcome_counts, labels=outcome_counts.index, colors=[PAIR_A, PAIR_B], startangle=90,
           counterclock=False, autopct=lambda p: f'{p:.1f}%\n({round(p * outcome_counts.sum() / 100)})')
    ax.set_title(f"{O['name']} outcome ({O['improved_label']} improved)")
    save_figure('0_cohort', f'outcome_pie_{outcome}')
    plt.show()

## Outcome class

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    outcome_class = (cohorts[outcome][O['scale']].map(O['classes'])
                     .value_counts().reindex(list(O['classes'].values()), fill_value=0))
    display(outcome_class)

    fig, ax = plt.subplots(figsize=(6, 4))
    # Improved classes teal, the rest orange
    colors = [PAIR_A if k <= O['improved_max'] else PAIR_B for k in O['classes']]
    bars = ax.bar(outcome_class.index, outcome_class.values, color=colors)
    ax.bar_label(bars)
    ax.set_xlabel(f"{O['name']} class ({O['improved_label'].split(' ', 1)[1]} improved, "
                  f"{O['not_improved_label'].split(' ', 1)[1]} not improved)")
    ax.set_ylabel('Patients')
    ax.set_title(f"{O['name']} outcome class (n={outcome_class.sum()})", loc='left')
    ax.spines[['top', 'right']].set_visible(False)
    save_figure('0_cohort', f'outcome_class_distribution_{outcome}')
    plt.show()